In [5]:
import pandas as pd 
import numpy as np
import scipy.stats as sci 


/Users/mienguyen/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# PnL Explain: Long Call

Goal: explain the 1-day PnL of a long call using Greeks, then compare with the actual PnL from full revaluation.

| Symbol | Meaning |
|---|---|
| S | Spot price |
| K | Strike |
| T | Time to maturity (years) |
| r | Risk-free rate |
| sigma | Volatility |

## 1. Greeks (call, computed at day 0)

In [6]:
# d1 = [ln(S/K) + (r + σ²/2)* T] / (σ√T)
# d2 = d1 − σ√T
# Call price = S * N(d1) − K * e^(-rT)* N(d2)
def bs_call_prices(S,K,T,r,sigma):
    d1 = (np.log(S/K) + ( r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    price = S * sci.norm.cdf(d1) - K * np.exp(-r*T) * sci.norm.cdf(d2)
    return price
bs_call_prices(100, 100, 30/365, 0.02, 0.20)

2.3683354165525543

In [7]:
# ATM call: Delta ≈ 0.5
def bs_call_delta(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    delta = sci.norm.cdf(d1)
    return delta
bs_call_delta (100,100,30/365, 0.02, 0.20)

0.5228621115361781

In [8]:
# Gamma = n(d1) / (S * σ * √T)
def bs_gamma(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    gamma = sci.norm.pdf(d1) / ( S * sigma * np.sqrt(T))
    return gamma
bs_gamma(100,100,30/365,0.02,0.20)



0.06946274895134844

In [9]:
# Vega per 1 vol point = S * n(d1) * √T / 100
def bs_vega(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    vega = S * sci.norm.pdf(d1) * np.sqrt(T)/100
    return vega
bs_vega(100,100,30/365,0.02,0.20)


0.11418534074194264

In [10]:
# Call Theta per day = [-S*n(d1)*σ/(2√T) - r*K*e^(-rT)*N(d2)] / 365
def bs_call_theta(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    theta = (-S * sci.norm.pdf(d1) * sigma / (2*np.sqrt(T)) - r * K * np.exp(-r*T)* sci.norm.cdf(d2))/365
    return theta
bs_call_theta(100,100,30/365,0.02,0.20)

-0.04079700631509862

In [11]:
# Call Rho per 1% rate = K*T*e^(-rT)*N(d2) / 100
def bs_call_rho(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T)) 
    d2 = d1 - sigma * np.sqrt(T)
    rho = K * T * np.exp(-r*T) * sci.norm.cdf(d2) /100
    return rho
bs_call_rho(100,100,30/365,0.02,0.20)

0.04102839101676596

## 2. Explained PnL (Delta + Gamma + Vega + Theta + Rho)

In [24]:
# LONG CALL
K, r, qty = 100, 0.02, 100
S0, S1 = 100, 102
vol0, vol1 = 0.20, 0.21
days_in_year = 365   # calendar-day convention (use 252 for trading days)
T0 = 30/days_in_year #time to maturity in years (30 days)
days = 1
r1 = 0.02

delta = bs_call_delta(S0, K, T0, r, vol0)
gamma = bs_gamma(S0, K, T0, r, vol0)
vega = bs_vega(S0, K, T0, r, vol0)
theta = bs_call_theta(S0, K, T0, r, vol0)
rho = bs_call_rho(S0, K, T0, r, vol0)

# PnL 
delta_PnL = delta * (S1 - S0) * qty
gamma_PnL = 1/2 * gamma * (S1-S0)**2 * qty
vega_PnL = vega * (vol1 - vol0)*100 * qty 
theta_PnL = theta * days * qty
rho_PnL = rho * (r1-r)*100 * qty

print(f"delta = {delta:.4f} | gamma = {gamma:.4f} | vega = {vega:.4f} |"
      f"theta = {theta:.4f} | rho = {rho:.4f}"
      )
print(f"delta_PnL = {delta_PnL: .4f} | gamma_PnL = {gamma_PnL: .4f} | vega_PnL = {vega_PnL: .4f} | "
      f"theta_PnL = {theta_PnL: .4f} | rho_PnL = {rho_PnL: .4f}")

explained = sum([delta_PnL, gamma_PnL, vega_PnL, theta_PnL, rho_PnL])
T1 = T0 - days/days_in_year
price0 = bs_call_prices(S0, K, T0, r, vol0)
price1 = bs_call_prices(S1, K, T1, r1, vol1)
actual = (price1 - price0) * qty 
unexplained = actual - explained
print( f"explained_pnl = {explained: .4f} | actual_pnl = {actual: .4f} | unexplained_pnl = {unexplained: .4f}")


delta = 0.5229 | gamma = 0.0695 | vega = 0.1142 |theta = -0.0408 | rho = 0.0410
delta_PnL =  104.5724 | gamma_PnL =  13.8925 | vega_PnL =  11.4185 | theta_PnL = -4.0797 | rho_PnL =  0.0000
explained_pnl =  125.8038 | actual_pnl =  124.7791 | unexplained_pnl = -1.0248


## 3. Scenario analysis: when does the Greeks approximation break down?

Goal: test how well Greeks explain the PnL of the long call across different market moves.

- Wrap the PnL explain into a function `pnl_explain(S1, vol1)`
- Run 4 scenarios (small move, down move, large up move, large down move)
- Compare Explained vs Actual PnL: how does Unexplained change?

In [23]:
def pnl_explained(S1,vol1):
    delta_PnL = delta * (S1 - S0) * qty
    gamma_PnL = 1/2 * gamma * (S1-S0)**2 * qty
    vega_PnL = vega * (vol1 - vol0)*100 * qty 
    theta_PnL = theta * days * qty
    rho_PnL = rho * (r1-r)*100 * qty
    explained = sum([delta_PnL, gamma_PnL, vega_PnL, theta_PnL, rho_PnL])

    T1 = T0 - days/days_in_year
    price1 = bs_call_prices(S1, K, T1, r1, vol1)
    actual = (price1 - price0) * qty
    unexplained = actual - explained
    return explained, actual, unexplained
scenarios = {
    "Base case": (100, 0.20),
    "Spot +2%, Vol + 1pt" : (102,0.21),
    "Spot -2%": (98,0.20),
    "Spot +10% (stress up)" : (110,0.20),
    "Spot -10% (stress down)" : (90,0.20)
}
for name, (S1_, vol1_) in scenarios.items():
    explained, actual, unexplained = pnl_explained(S1_,vol1_)
    print(f"{name:25s} | Explained_pnl = {explained:8.2f} | Actual_pnl={actual:8.2f} | "
          f"Unexplained_pnl = {unexplained:8.2f}")


Base case                 | Explained_pnl =    -4.08 | Actual_pnl=   -4.11 | Unexplained_pnl =    -0.03
Spot +2%, Vol + 1pt       | Explained_pnl =   125.80 | Actual_pnl=  124.78 | Unexplained_pnl =    -1.02
Spot -2%                  | Explained_pnl =   -94.76 | Actual_pnl=  -94.44 | Unexplained_pnl =     0.32
Spot +10% (stress up)     | Explained_pnl =   866.10 | Actual_pnl=  789.36 | Unexplained_pnl =   -76.74
Spot -10% (stress down)   | Explained_pnl =  -179.63 | Actual_pnl= -229.96 | Unexplained_pnl =   -50.33


In [26]:
print(bs_gamma(100, K, T0, r, vol0))
print(bs_gamma(110, K, T0, r, vol0))
print(bs_gamma(90,  K, T0, r, vol0))

0.06946274895134844
0.014420353462641883
0.01585142130521138


## Scenario analysis: 
- Black-Scholes, European call, no dividends, 365-day calendar convention.
- Position: long 100 calls (K = 100, S0 = 100, 30 days to maturity), 1-day horizon.

| Scenario | Explained | Actual | Unexplained | Unexplained / Actual |
|---|---|---|---|---|
| Base case (time only) | -4.08 | -4.11 | -0.03 | 0.7% |
| Spot +2%, Vol +1pt | 125.80 | 124.78 | -1.02 | 0.8% |
| Spot -2% | -94.76 | -94.44 | 0.32 | 0.3% |
| Spot +10% (stress up) | 866.10 | 789.36 | -76.74 | 9.7% |
| Spot -10% (stress down) | -179.63 | -229.96 | -50.33 | 21.9% |

### Observations
- **Small moves (up to +/- 2%):** Greeks explain the PnL almost fully (Unexplained below 1% of Actual). In the base case, the loss is entirely Theta (time decay).
- **Large moves (+/-10%):** Unexplained grows disproportionately (about 10% to 22% of Actual)
- Greeks are local apporximation at S0. Gamma peaks ATM (0.0695 at S =100) and falls sharply away from the strike (0.0144 at S = 110, 0.0159 at S = 90). The Gamma PnL term assumes it stays constant (fixed at S0), but in reality Gamma changes as spot moves. The approximation therefore overstates the Gamma gain on both sides.
- Greeks-based estimates are too optimistic for large moves (they understate the loss in the stress-down scenario (explained = -180 vs actual = - 230). Stress tests and tail risk mesures should use full revaluation.
